# 응용 모의고사 Set 6 — 정답 — 단계형 전처리와 분류

- 데이터: `edu_enrollees.csv`
- 난이도: 기존 Set 01~06과 유사
- 구성: **공통 전처리 → Q1 통계 → Q2 상관분석 → Q3 모델링**
- 모든 문항은 공통 전처리 결과를 이어서 사용합니다.
- 전처리 완료 후 데이터는 **7,439행**이어야 합니다. 행 수가 다르면 다음 문제로 넘어가기 전에 전처리를 확인하세요.

정답 노트북은 `../answers/`에 있습니다.

## 공통 전처리 정답

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('../../dataset/edu_enrollees.csv')
base = df.drop(columns=['city', 'company_size', 'company_type']).copy()
object_cols = base.select_dtypes(include='object').columns
base = base.dropna(subset=object_cols).copy()
base = base.loc[~base['experience'].isin(['>20', '<1'])].copy()
base['experience'] = base['experience'].astype(int)
base = base.loc[~base['last_new_job'].isin(['>4', 'never'])].copy()
base['last_new_job'] = base['last_new_job'].astype(int)
base = base.loc[base['education_level'] != 'Primary School'].copy()
base['education_level'] = base['education_level'].replace(
    {'Graduate': 1, 'Masters': 2, 'Phd': 3}
)
base = base.loc[base['gender'] != 'Other'].copy()
base['target'] = base['target'].astype(int)
assert len(base) == 7439
display(base.head())

## Q1 정답

In [ ]:
target_rate = base.groupby('gender')['target'].mean()
answer_q1 = round(target_rate.loc['Female'] / target_rate.loc['Male'], 2)
display(target_rate, answer_q1)  # 0.94

## Q2 정답

In [ ]:
from sklearn.linear_model import LogisticRegression

categorical = ['gender', 'relevant_experience',
               'enrolled_university', 'major_discipline']
numeric = ['city_development_index', 'education_level', 'experience',
           'last_new_job', 'training_hours']
dummy_parts = []
for col in categorical:
    categories = sorted(base[col].unique())
    dummy = pd.get_dummies(base[col], prefix=col)
    dummy = dummy.drop(columns=f'{col}_{categories[0]}')
    dummy_parts.append(dummy)
X = pd.concat([base[numeric]] + dummy_parts, axis=1)
y = base['target']
model = LogisticRegression(
    C=100000, max_iter=1000, solver='liblinear', random_state=321
)
model.fit(X, y)
odds_ratio = pd.Series(np.exp(model.coef_[0]), index=X.columns)
answer_var_q2 = odds_ratio.idxmax()
answer_or_q2 = np.floor(odds_ratio.max() * 100) / 100
display(odds_ratio.sort_values(ascending=False), answer_var_q2, answer_or_q2)
# relevant_experience_No relevant experience, 2.17

## Q3 정답

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler

job = X.copy()
job['target'] = y
job['Xgrp'] = base['Xgrp']
train = job.loc[job['Xgrp'] == 'train'].copy()
test = job.loc[job['Xgrp'] == 'test'].copy()
X_train = train.drop(columns=['target', 'Xgrp'])
y_train = train['target']
X_test = test.drop(columns=['target', 'Xgrp'])
y_test = test['target']
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
knn = KNeighborsClassifier(n_neighbors=7, metric='euclidean')
knn.fit(X_train_scaled, y_train)
pred = knn.predict(X_test_scaled)
answer_q3 = round(accuracy_score(y_test, pred), 2)
display(answer_q3)  # 0.75